# Test Redshift Connection

⚠️ **IMPORTANT**: Change `USER_NAME` below to your name to avoid overwriting other students' data!

In [ ]:
import boto3, time
from datetime import date

# ⚠️ TODO: CHANGE THIS TO YOUR NAME (to avoid overwriting other students' data)
USER_NAME = "student_name"  # ⚠️ CHANGE THIS! Example: "john_doe"

REDSHIFT_CONFIG = {
    'cluster_id': 'redshift-cluster-dsi',
    'database': 'prod',
    'db_user': 'svc_sagemaker',
    'region': 'af-south-1',
    'iam_role': 'arn:aws:iam::733246370304:role/RedshiftIAMAuthRole',
    's3_export_prefix': f"s3://sagemaker-af-south-1-733246370304/redshift_exports/{USER_NAME}/prod_train_{date.today():%Y-%m-%d}/"
}

def run_redshift_query(sql):
    client = boto3.client("redshift-data", region_name=REDSHIFT_CONFIG["region"])
    resp = client.execute_statement(
        ClusterIdentifier=REDSHIFT_CONFIG["cluster_id"],
        Database=REDSHIFT_CONFIG["database"],
        DbUser=REDSHIFT_CONFIG["db_user"],
        Sql=sql
    )
    stmt_id = resp["Id"]
    while True:
        desc = client.describe_statement(Id=stmt_id)
        if desc["Status"] in ["FINISHED", "FAILED", "ABORTED"]:
            break
        time.sleep(2)
    return desc

print(f"✓ Configuration loaded")
print(f"  User: {USER_NAME}")
print(f"  S3 Export Path: {REDSHIFT_CONFIG['s3_export_prefix']}")

In [2]:
# build UNLOAD
sql = f"""
UNLOAD ('
  SELECT *
  FROM dth_churn_ml_training.training_features
  WHERE RANDOM() < 0.0001;
')
TO '{REDSHIFT_CONFIG["s3_export_prefix"]}'
IAM_ROLE '{REDSHIFT_CONFIG["iam_role"]}'
FORMAT AS PARQUET
ALLOWOVERWRITE
PARALLEL ON
REGION '{REDSHIFT_CONFIG["region"]}';
"""

In [3]:
desc = run_redshift_query(sql)
print(desc["Status"])

FINISHED


In [ ]:
import awswrangler as wr

prefix = REDSHIFT_CONFIG["s3_export_prefix"]
files = wr.s3.list_objects(prefix)
print(f"Found {len(files)} objects at {prefix}")
for p in files[:10]:
    print(p)

In [6]:
# Read the dataset into a DataFrame
df = wr.s3.read_parquet(prefix, dataset=True)

print(f"✅ Loaded {len(df):,} rows and {len(df.columns)} columns")
df.head()

✅ Loaded 3,774 rows and 58 columns


,idconsumo,id_contaservico,codigocontaservico,idconta,iddim_date_inicio,iddim_date_fim,id_produto_actual,tipo_produto_actual,tipo_subscricao,tipo_stb,...,was_contacted,topup_count,topup_total_value,topup_avg_value,topup_std_value,topup_cv_value,topup_days_since_last,used_selfcare,topup_type_nunique,topup_channel_nunique
0,513883031,1625728,100022230201,1574428,2025-05-27,2025-06-09,24,tafacil7,7,HD,...,0,13,14736.84,1133.60,340.605335,0.300463,7,0,2,1
1,534067661,15792,100137810101,14474,2025-08-02,2025-09-30,22,normal,7,DVR,...,0,7,22350.88,3192.98,4224.380682,1.323021,30,0,1,1
2,526398740,4383379,100335960301,4342746,2025-07-08,2025-07-10,24,tafacil7,7,HD,...,0,29,14438.60,497.88,744.532451,1.495405,2,0,3,1
3,520331991,4726512,100457680301,4692093,2025-06-18,2025-06-22,22,normal,7,HD,...,0,35,21631.55,618.04,1070.394583,1.731918,4,0,3,1
4,532165068,55555,100480850101,52274,2025-07-28,2025-07-30,24,tafacil7,7,DVR,...,0,10,13771.92,1377.19,1709.862116,1.241559,2,0,2,1
